<a href="https://colab.research.google.com/github/Sidnaik04/AI-Engineering/blob/main/Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embeddings in LLM

### One-hot encoding

In [ ]:
# Implementaion using Pandas

import pandas as pd

data = {
    'Employee_ID': [10, 20, 15, 25, 30],
    'Gender': ['M', 'F', 'F', 'M', 'F'],
    'Remarks': ['Good', 'Nice', 'Good', 'Great', 'Nice']
}

df = pd.DataFrame(data)

print("Original Data:")
print(df)

encoded_df = pd.get_dummies(
    df,
    columns=['Gender', 'Remarks'],
    drop_first=True
)

print("\nOne-Hot Encoded Data:")
print(encoded_df)

Original Data:
   Employee_ID Gender Remarks
0           10      M    Good
1           20      F    Nice
2           15      F    Good
3           25      M   Great
4           30      F    Nice

One-Hot Encoded Data:
   Employee_ID  Gender_M  Remarks_Great  Remarks_Nice
0           10      True          False         False
1           20     False          False          True
2           15     False          False         False
3           25      True           True         False
4           30     False          False          True


In [ ]:
# Implementation using Scikit-Learn

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

data = {
    'Employee_ID': [10, 20, 15, 25, 30],
    'Gender': ['M', 'F', 'F', 'M', 'F'],
    'Remarks': ['Good', 'Nice', 'Good', 'Great', 'Nice']
}

df = pd.DataFrame(data)

print("Original Data:")
print(df)

categorical_columns = df.select_dtypes(include=['object']).columns

encoder = OneHotEncoder(sparse_output=False)

encoded_data = encoder.fit_transform(df[categorical_columns])

encoded_df = pd.DataFrame(
    encoded_data,
    columns = encoder.get_feature_names_out(categorical_columns)
)

final_df = pd.concat(
    [df.drop(columns=categorical_columns), encoded_df],
    axis=1
)

print("\nOne-Hot encoded Data:")
print(final_df)

Original Data:
   Employee_ID Gender Remarks
0           10      M    Good
1           20      F    Nice
2           15      F    Good
3           25      M   Great
4           30      F    Nice

One-Hot encoded Data:
   Employee_ID  Gender_F  Gender_M  Remarks_Good  Remarks_Great  Remarks_Nice
0           10       0.0       1.0           1.0            0.0           0.0
1           20       1.0       0.0           0.0            0.0           1.0
2           15       1.0       0.0           1.0            0.0           0.0
3           25       0.0       1.0           0.0            1.0           0.0
4           30       1.0       0.0           0.0            0.0           1.0


---

### Word2Vec

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 33.9 MB/s eta 0:00:00


In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# import libaries

import gensim
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download('punkt')
import warnings

warnings.filterwarnings(action='ignore')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# loading and cleaning dataset

import zipfile

with zipfile.ZipFile("/content/Gutenburg.zip", "r") as zip_ref:
  file_name = zip_ref.namelist()[0]

  with zip_ref.open(file_name) as file:
    content = file.read().decode('utf-8', errors='ignore')
    cleaned_text = content.replace('\n', ' ')
    print("File loaded")

File loaded


In [ ]:
# text tokenization

data = []

for i in sent_tokenize(cleaned_text):
  temp = []

  for j in word_tokenize(i):
    temp.append(j.lower())

  data.append(temp)

##### building word2vec model

In [ ]:
# using CBOW architecture
model1 = gensim.models.Word2Vec(data, min_count=1, vector_size=100, window=5)

# using skip-gram architecture
model2 = gensim.models.Word2Vec(data, min_count=1, vector_size=100, window=5, sg=1)

In [ ]:
# evaluating word similarities

print("Cosine similarity between 'alice' " +"and 'wonderland' - CBOW : ", model1.wv.similarity('alice', 'wonderland'))

print("Cosine similarity between 'alice' " +"and 'machines' - CBOW : ", model1.wv.similarity('alice', 'machines'))

Cosine similarity between 'alice' and 'wonderland' - CBOW :  0.9810152
Cosine similarity between 'alice' and 'machines' - CBOW :  0.9664741


In [ ]:
print("Cosine similarity between 'alice' " +"and 'wonderland' - Skip-gram : ", model2.wv.similarity('alice', 'wonderland'))

print("Cosine similarity between 'alice' " +"and 'machines' - Skip-gram : ", model1.wv.similarity('alice', 'machines'))

Cosine similarity between 'alice' and 'wonderland' - Skip-gram :  0.8896627
Cosine similarity between 'alice' and 'machines' - Skip-gram :  0.9664741


---

### BERT (Bidirectional Encoder Representations From Transformers)

In [ ]:
!pip install transformers

In [ ]:
# import libraries
import random
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# random seed
random_seed = 42
random.seed(random_seed)

# set a random seed for pytorch (for GPU as well)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(random_seed)

In [ ]:
# loading bert pre-trained model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# tokenize and encode text

# input text
text = "Learning and building AI and Software is fascinating"

# Tokenize and encode text using batch_encode_plus
# The function returns a dictionary containing the token IDs and attention masks
encoding = tokenizer( [text],# List of input texts
    padding=True,              # Pad to the maximum sequence length
    truncation=True,           # Truncate to the maximum sequence length if necessary
    return_tensors='pt',      # Return PyTorch tensors
    add_special_tokens=True    # Add special tokens CLS and SEP
)

input_ids = encoding['input_ids'] # token IDs

print(f"Input ID: {input_ids}")

attention_mask = encoding['attention_mask']

print(f"Attention mask: {attention_mask}")

Input ID: tensor([[  101,  4083,  1998,  2311,  9932,  1998,  4007,  2003, 17160,   102]])
Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [ ]:
# Generate embeddings for the input sentence
with torch.no_grad():
    output = model(input_ids, attention_mask=attention_mask)
    sentence_embedding = output.last_hidden_state.mean(dim=1)

In [ ]:
# generate word embeddings

with torch.no_grad():
  outputs = model(input_ids, attention_mask=attention_mask)
  word_embeddings = outputs.last_hidden_state

print(f"Shape of Word Embeddings: {word_embeddings.shape}")

Shape of Word Embeddings: torch.Size([1, 10, 768])


In [ ]:
# decode and encode the text

# Decode the token IDs back to text
decoded_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print(f"Decoded Text: {decoded_text}")

# Tokenize the text again for reference
tokenized_text = tokenizer.tokenize(decoded_text)
print(f"Tokenized Text: {tokenized_text}")

# Encode the text
encoded_text = tokenizer.encode(text, return_tensors='pt')  # Returns a tensor
print(f"Encoded text: {encoded_text}")

Decoded Text: learning and building ai and software is fascinating
Tokenized Text: ['learning', 'and', 'building', 'ai', 'and', 'software', 'is', 'fascinating']
Encoded text: tensor([[  101,  4083,  1998,  2311,  9932,  1998,  4007,  2003, 17160,   102]])


In [ ]:
# computing similarity metrics

example_sentence = "Software development with AI is interesting and fast"

# Tokenize and encode the example sentence
example_encoding = tokenizer(
    [example_sentence],
    padding=True,
    truncation=True,
    return_tensors='pt',
    add_special_tokens=True
)

example_input_ids = example_encoding['input_ids']
example_attention_mask = example_encoding['attention_mask']

# Generate embeddings for the example sentence
with torch.no_grad():
    example_outputs = model(example_input_ids, attention_mask=example_attention_mask)
    example_sentence_embedding = example_outputs.last_hidden_state.mean(dim=1)

# Compute cosine similarity between the original sentence embedding and the example sentence embedding
similarity_score = cosine_similarity(sentence_embedding, example_sentence_embedding)

print("Cosine Similarity Score: ", similarity_score[0][0])

Cosine Similarity Score:  0.8991295


---

### Measuring Similarity

In [ ]:
# NumPy Implementation

import numpy as np

# Two example embeddings (d = 4)
u = np.array([1.0, 2.0, 3.0, 0.0])
v = np.array([2.0, 4.0, 5.0, 1.0])

# 1. Dot Product
dot_product = np.dot(u, v)

# 2. Cosine Similarity
norm_u = np.linalg.norm(u)
norm_v = np.linalg.norm(v)
cosine_sim = dot_product / (norm_u * norm_v)

# 3. Euclidean Distance
euclidean_dist = np.linalg.norm(u - v)

print(f"Dot Product        : {dot_product:.4f}")
print(f"Cosine Similarity  : {cosine_sim:.4f}")
print(f"Euclidean Distance : {euclidean_dist:.4f}")

Dot Product        : 25.0000
Cosine Similarity  : 0.9851
Euclidean Distance : 3.1623


In [ ]:
# Pytorch Implementation

import torch
import torch.nn.functional as F

u = torch.tensor([1.0, 2.0, 3.0, 0.0])
v = torch.tensor([2.0, 4.0, 5.0, 1.0])

# 1. Dot Product
dot_product = torch.dot(u, v)

# 2. Cosine Similarity (Built-in PyTorch function)
# PyTorch expects batch dimensions [1, d]
cosine_sim = F.cosine_similarity(u.unsqueeze(0), v.unsqueeze(0))

# 3. Euclidean Distance
euclidean_dist = torch.pairwise_distance(u.unsqueeze(0), v.unsqueeze(0))

print(f"PyTorch Dot Product       : {dot_product.item():.4f}")
print(f"PyTorch Cosine Similarity : {cosine_sim.item():.4f}")
print(f"PyTorch Euclidean Dist    : {euclidean_dist.item():.4f}")

# Pro-Tip: Cosine via Dot Product on Normalized Vectors
u_norm = F.normalize(u, p=2, dim=0)
v_norm = F.normalize(v, p=2, dim=0)
cosine_via_dot = torch.dot(u_norm, v_norm)
print(f"Dot product on L2-normalized vectors: {cosine_via_dot.item():.4f}")

PyTorch Dot Product       : 25.0000
PyTorch Cosine Similarity : 0.9851
PyTorch Euclidean Dist    : 3.1623
Dot product on L2-normalized vectors: 0.9851


---

### OpenAI Embedding Models

In [ ]:
!pip install openai

In [8]:
# import libraries

import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI()

In [9]:
# function to get embeddings

def get_embedding(text, model='text-embedding-3-small'):
  """
  Generate embeddings for the given text using OpenAI's API.

  Args:
  text (str): Input text to embed
  model (str): OpenAI embedding model to use

  Returns:
  list: Embedding vector
  """
  text = text.replace('\n',' ')
  response = client.embeddings.create(input=[text], model=model)
  return response.data[0].embedding

In [10]:
# example dataset

# Sample product descriptions
products = [
  "Wireless noise-canceling headphones with 30-hour battery life",
  "Ergonomic office chair with lumbar support and adjustable armrests",
  "Stainless steel water bottle with vacuum insulation, keeps drinks cold for 24 hours",
  "Portable Bluetooth speaker with waterproof design and 360-degree sound",
  "Standing desk converter with dual-tier design for monitor and keyboard"
]

In [11]:
# generate embeddings

product_embeddings = []

for product in products:
  embedding = get_embedding(product)
  product_embeddings.append(embedding)
  print(f"Generated embedding for: {product[:50]}")

# Convert to numpy array for easier computation
product_embeddings = np.array(product_embeddings)
print(f"\nShape of embeddings: {product_embeddings.shape}")

Generated embedding for: Wireless noise-canceling headphones with 30-hour b
Generated embedding for: Ergonomic office chair with lumbar support and adj
Generated embedding for: Stainless steel water bottle with vacuum insulatio
Generated embedding for: Portable Bluetooth speaker with waterproof design 
Generated embedding for: Standing desk converter with dual-tier design for 

Shape of embeddings: (5, 1536)


In [12]:
# similarity search

# user query
query = "headphones for music listening"

# generate embedding for query
query_embedding = get_embedding(query)
query_embedding = np.array(query_embedding).reshape(1,-1)

# calculate cosine similarity between query and all products
similarities = cosine_similarity(query_embedding, product_embeddings)[0]

# rank products by similarities
ranked_indices = np.argsort(similarities)[::-1]

print(f"\nSearch results for: '{query}'\n")

for idx in ranked_indices:
  print(f"{similarities[idx]:.4f} - {products[idx]}")


Search results for: 'headphones for music listening'

0.5007 - Wireless noise-canceling headphones with 30-hour battery life
0.3820 - Portable Bluetooth speaker with waterproof design and 360-degree sound
0.2190 - Standing desk converter with dual-tier design for monitor and keyboard
0.1878 - Ergonomic office chair with lumbar support and adjustable armrests
0.1698 - Stainless steel water bottle with vacuum insulation, keeps drinks cold for 24 hours


In [14]:
# openai response

from openai import OpenAI

client = OpenAI()

response = client.embeddings.create(
    input="This is fun right?", model='text-embedding-3-small'
)

print(response.data[0].embedding)
print(response)

[0.016143798828125, 0.036773681640625, 0.0031948089599609375, 0.00970458984375, -0.00885772705078125, -0.0196533203125, -0.0296173095703125, 0.0086212158203125, -0.0173797607421875, 0.021270751953125, 0.006916046142578125, -0.0245361328125, -0.005321502685546875, 0.00548553466796875, 0.0119171142578125, 0.0174713134765625, -0.06927490234375, -0.0120086669921875, 0.0077362060546875, 0.0017442703247070312, 0.023406982421875, -0.00780487060546875, 0.0177459716796875, 0.034759521484375, 0.0169830322265625, -0.035797119140625, -0.0201873779296875, -0.002376556396484375, 0.041229248046875, -0.00833892822265625, 0.0274505615234375, -0.02630615234375, 0.0099029541015625, -0.0281524658203125, 0.052734375, 0.0252685546875, -0.0235137939453125, -0.031494140625, -0.036895751953125, 0.0145416259765625, 0.028167724609375, 0.020782470703125, 0.014739990234375, 0.01239776611328125, -0.0435791015625, -0.031524658203125, -0.053131103515625, -0.0159759521484375, -0.02056884765625, 0.00501251220703125, -0

---

### BGE (BAAI General Embedding) Models

In [15]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Load BGE-Base v1.5 Model and Tokenizer
model_name = "BAAI/bge-base-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# BGE requires an instruction prefix for QUERIES only (not passages/documents)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

query = QUERY_PREFIX + "How do I optimize vector search?"
documents = [
    "HNSW indexes allow fast approximate nearest neighbor search.",
    "BGE embeddings are trained using multi-stage contrastive learning.",
    "Python is a popular programming language for software engineering."
]

texts = [query] + documents

# Tokenized inputs
encoded_inputs = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

# compute embeddings
with torch.no_grad():
  model_output = model(**encoded_inputs)
  # Perform CLS Pooling (BGE uses [CLS] token representation)
  sentence_embeddings = model_output[0][:, 0]

# L2 Normalize embeddings
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

# extract query and document vectors
query_vec = sentence_embeddings[0]
doc_vecs = sentence_embeddings[1:]

# Compute Cosine Similarity via Dot Product (since normalized)
similarities = torch.matmul(doc_vecs, query_vec)

print("Query:", query.replace(QUERY_PREFIX, ""))
for doc, score in zip(documents, similarities):
    print(f"  Score: {score.item():.4f} | Document: '{doc}'")

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Query: How do I optimize vector search?
  Score: 0.5855 | Document: 'HNSW indexes allow fast approximate nearest neighbor search.'
  Score: 0.5136 | Document: 'BGE embeddings are trained using multi-stage contrastive learning.'
  Score: 0.3865 | Document: 'Python is a popular programming language for software engineering.'


In [16]:
# sentence-transformers - (for production RAG)

from sentence_transformers import SentenceTransformer

# Load BGE model via Sentence Transformers
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

query = "What is the capital of France?"
passages = [
    "Paris is the capital and most populous city of France.",
    "Berlin is the capital of Germany.",
    "The Eiffel Tower is located in Paris."
]

query_embedding = model.encode(
    query,
    prompt="Represent this sentence for searching relevant passages: ",
    normalize_embeddings=True
)

# Passages are encoded WITHOUT instruction prompt
passage_embeddings = model.encode(passages, normalize_embeddings=True)

# Compute Similarity Scores
scores = query_embedding @ passage_embeddings.T

for passage, score in zip(passages, scores):
    print(f"Score: {score:.4f} | Passage: '{passage}'")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Score: 0.7253 | Passage: 'Paris is the capital and most populous city of France.'
Score: 0.5314 | Passage: 'Berlin is the capital of Germany.'
Score: 0.6163 | Passage: 'The Eiffel Tower is located in Paris.'


---

### E5 Embedding Model

In [18]:
import torch
from sentence_transformers import SentenceTransformer

# Load E5-Base-v2
model = SentenceTransformer("intfloat/e5-base-v2")

query = "What is the capital of France?"
passages = [
    "Paris is the capital and most populous city of France.",
    "Berlin is the capital of Germany.",
    "The Eiffel Tower is located in Paris."
]

# Step 1: Format with REQUIRED prefixes
formatted_query = f"query: {query}"
formatted_passages = [f"passage: {p}" for p in passages]

# Step 2: Encode with L2 Normalization
query_emb = model.encode(formatted_query, normalize_embeddings=True)
passage_embs = model.encode(formatted_passages, normalize_embeddings=True)

# Step 3: Compute Cosine Similarity via Dot Product
scores = query_emb @ passage_embs.T

for passage, score in zip(passages, scores):
    print(f"Score: {score:.4f} | Passage: {passage}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Score: 0.8518 | Passage: Paris is the capital and most populous city of France.
Score: 0.7505 | Passage: Berlin is the capital of Germany.
Score: 0.7620 | Passage: The Eiffel Tower is located in Paris.
